# Medical Devices API -> pandas DataFrame
Open this notebook from the cloned project in a Databricks Git folder. All data is held in driver memory; no Volume, wheel upload, Spark or table is required. The result is `devices_df`, a pandas DataFrame.
Use the entire version 0.2.0 project, not a notebook-only copy. The public function is fetch_medical_devices_dataframe, defined in src/moh_medical_devices/pandas_ingestion.py. This project contains no Spark/table ingestion path.

In [ ]:
%pip install "requests>=2.32.3,<3" "pandas>=2.2,<3"

Restart Python after installation if requested by the runtime. include_details=True fetches all detail rows once per registration. The notebook defaults to 4 worker threads with a shared limit of 2 request starts per second, including retries. Each worker has its own Session. HTTP 429/5xx cooldown is shared; access errors stop the run. At about 30,000 registrations, the rate-limit floor alone is about 4.2 hours; network latency, retries and server limits add time. This rate is a client setting, not an approved server quota. Set detail_workers=1 for the original sequential mode (delay_seconds applies there), or include_details=False for listing fields only.

In [ ]:
from pathlib import Path
import importlib
import logging
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/moh_medical_devices').is_dir()), None)
if project_root is not None:
    package_dir = project_root / 'src/moh_medical_devices'
    required_files = ['__init__.py', 'client.py', 'pandas_ingestion.py', 'record_normalization.py']
    missing = [name for name in required_files if not (package_dir / name).is_file()]
    if missing:
        raise RuntimeError(f'Incomplete project checkout; missing {missing}. Update the entire project, not only this notebook.')
    sys.path.insert(0, str(project_root / 'src'))
importlib.invalidate_caches()

try:
    import moh_medical_devices
    from moh_medical_devices import MedicalDevicesClient, MedicalDevicesConfig, fetch_medical_devices_dataframe
except (ModuleNotFoundError, ImportError) as exc:
    raise RuntimeError('Update the complete project to version 0.2.0 and restart Python. Open this notebook from the project Git folder, or pip install the complete project.') from exc

if getattr(moh_medical_devices, '__version__', None) != '0.2.0':
    raise RuntimeError('An older package is loaded. Update the entire project, restart Python and rerun from the first cell.')
print('Loaded:', moh_medical_devices.__file__, 'version:', moh_medical_devices.__version__)
print('Extraction function:', fetch_medical_devices_dataframe.__module__ + '.' + fetch_medical_devices_dataframe.__name__)

logging.basicConfig(level=logging.INFO)
page_size = 1000
include_details = True
search = ''
declaration_only = False
delay_seconds = 2
detail_workers = 4
requests_per_second = 2

In [ ]:
import sys
import traceback

# Clear a previous result and diagnostic before each run.
devices_df = None
last_error = None
last_error_traceback = None
client = None
try:
    try:
        client = MedicalDevicesClient(MedicalDevicesConfig(delay_seconds=delay_seconds))
        devices_df = fetch_medical_devices_dataframe(
            client, page_size=page_size, include_details=include_details,
            search=search, declaration_only=declaration_only,
            detail_workers=detail_workers, requests_per_second=requests_per_second,
        )
    finally:
        if client is not None:
            client.close()
except Exception as exc:
    last_error = exc
    last_error_traceback = ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__, chain=True))
    print(last_error_traceback, file=sys.stderr, end='')
    raise

print(devices_df.attrs['extraction'])
display(devices_df.head(20))

## Result
`devices_df` retains each source listing row, including duplicate registration numbers, normalized columns, all original fields and raw_json. `details` is a list of all rows returned by the detail endpoint; details_raw_json retains the complete detail response. The detail fetch time is recorded separately. No files or tables are written. This DataFrame lives only in the current Python session; an interrupted run must restart from the beginning. The API has no verified snapshot token, so same-count changes outside the first page cannot be ruled out. An API/detail failure raises an exception rather than returning partial success.
On failure, the complete chained traceback is printed and the same exception is re-raised. last_error retains the exception object; last_error_traceback retains its untruncated formatted traceback, including cause/context and exception notes. HTTP failures also retain last_error.response when the error is a RegistryError raised for a response, including the full response body. Request headers/cookies are not printed. Treat diagnostic output as potentially containing upstream data.